# Free embeddings and custom embeddings

Check out [embedding providers](https://docs.lancedb.com/embedding/quickstart) in lancedb

## How to read this notebook (map)

The previous notebook, `lancedb_basics.ipynb`, used a **paid, hosted** embedding model (Cohere). This one shows two other ways to make embeddings.

| Option | Where the model runs | Key needed | Cell numbers (0 = title) |
| --- | --- | --- | --- |
| A. `sentence-transformers` (`intfloat/multilingual-e5-small`) | on **your computer**, free | none | 4 - 6 |
| B. Custom NVIDIA Nemotron embedder | NVIDIA API (internet) | `NVIDIA_API` in `.env` | 8 - 10 |

**Files this notebook uses**

| File / folder | Role |
| --- | --- |
| `jokes.json` | the 30 jokes (read with pandas) |
| `custom_embeddings/nvidia_nemotron.py` | our own embedding class (imported in the Option B cell) |
| `knowledge_base/` | the database folder created by this notebook (tables `jokes_sentence_transformer`, `jokes_nvidia`) |
| `.env` | `NVIDIA_API` for option B |

```
jokes.json --pandas--> df --table.add()--> [embedding model] --> table (jokes + vector) --search("text")--> nearest jokes
                                                ^
                          A: sentence-transformers (local)   or   B: nvidia_nemotron.py -> NVIDIA API
```

# Imports and set-up
- jokes.json is read with pandas: it becomes a DataFrame with ONE column called "jokes" (plural).
- lancedb.connect("knowledge_base") opens/creates a NEW database folder, separate from
- the "vector_database" folder of lancedb_basics.ipynb.

In [ ]:

import lancedb
import pandas as pd 

df = pd.read_json("jokes.json")

db = lancedb.connect("knowledge_base")
df.head(), db

(                                               jokes
 0  Parallel lines have so much in common—it’s sad...
 1  ETL stands for “Extract, Transform, Leave for ...
 2  What do you call a snake that runs your script...
 3  Gold walks into a bar. The bartender says, “Au...
 4   C# devs don’t argue; they just throw exceptions.,
 LanceDBConnection(uri='c:\\Users\\adelo\\de25\\ai_ops_lecture\\09_lancedb\\knowledge_base'))

## Sentence transformer

It will download the model first time you use it

In [2]:
# OPTION A - a FREE embedding model that runs on YOUR computer (no API key, no cost).
# "sentence-transformers" is a Python library that downloads a model from Hugging Face the first time.
# intfloat/multilingual-e5-small: small, understands many languages, outputs 384 numbers per text.
# device="cpu": run on the CPU (use "cuda" if you have an NVIDIA GPU).
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

model = (
    get_registry()
    .get("sentence-transformers")
    .create(name="intfloat/multilingual-e5-small", device="cpu")
)


class JokeModel(LanceModel):
    jokes: str = model.SourceField()  # marks text as input to embedding function
    # ^ the field name must match the DataFrame column name, which is `jokes` here
    vector: Vector(model.ndims()) = (
        model.VectorField()
    )  # vector is destination of computed embedding


# Create the table. (No exist_ok/mode here, so re-running this cell fails if the table already exists.)
table = db.create_table("jokes_sentence_transformer", schema=JokeModel)
# Add the DataFrame: LanceDB embeds every joke locally and fills the `vector` column.
table.add(df)

c:\Users\adelo\de25\ai_ops_lecture\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\adelo\de25\ai_ops_lecture\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\adelo\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In ord

AddResult(version=2)

## Text Search

In [3]:
# Text search: the query is embedded with the same model, then nearest jokes come back.
# "mathematicians" finds the '6 afraid of 7' joke even though that word is not in it: meaning, not keywords.
table.search("why do mathematicians have problems?").to_pandas()

,jokes,vector,_distance
0,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.273110
1,Why do data engineers hate nature? Too many un...,"[0.057509, 0.012957, -0.081348, -0.042842, 0.0...",0.281742
2,My math teacher said I’m average… how mean!,"[-0.000705, 0.014578, -0.084753, -0.031266, 0....",0.287475
3,Why did the chemist ground his kids? Because t...,"[-0.000735, -0.029147, -0.051178, -0.053038, 0...",0.300044
4,"Dear algebra, please stop asking us to find yo...","[0.05175, 0.00825, -0.020384, -0.038882, 0.084...",0.310816
5,C# devs don’t argue; they just throw exceptions.,"[0.028011, 0.004646, -0.057664, -0.064801, 0.0...",0.316030
6,Why did the C# developer go broke? He kept usi...,"[0.020794, 0.007793, -0.040906, -0.02401, 0.03...",0.318841
7,Parallel lines have so much in common—it’s sad...,"[0.037198, -0.019918, -0.051841, -0.032693, 0....",0.326532
8,"Data engineer motto: If it works, don’t touch ...","[0.003134, -0.036774, -0.050577, -0.060755, 0....",0.331513
9,Why did the Python programmer get bitten? Beca...,"[0.029594, 0.007605, -0.018481, -0.004575, 0.0...",0.334075


In [4]:
# A query in another language (a Danish/Swedish-style word for "chemistry joke").
# Because the model is MULTILINGUAL, chemistry jokes in English still come out on top.
table.search("kemi skämt").to_pandas()

,jokes,vector,_distance
0,Chemists have all the solutions… mostly in bea...,"[0.025907, -0.022183, -0.06204, -0.036586, 0.0...",0.357881
1,Why did the chemist ground his kids? Because t...,"[-0.000735, -0.029147, -0.051178, -0.053038, 0...",0.361318
2,Oxygen and Magnesium got together—OMg!,"[0.050211, -0.010728, -0.099408, -0.035509, 0....",0.372850
3,I told a chemistry joke… there was no reaction.,"[0.048405, 0.011754, -0.043688, -0.029928, 0.1...",0.374648
4,"My pipeline failed again, so I’m logging out—l...","[0.014553, -0.021881, 0.002843, -0.057108, 0.0...",0.391870
5,What do you call a snake that runs your script...,"[0.073168, -0.00455, -0.027777, -0.054148, 0.0...",0.398163
6,My math teacher said I’m average… how mean!,"[-0.000705, 0.014578, -0.084753, -0.031266, 0....",0.405061
7,Why’s 6 afraid of 7? Because 7 8 9.,"[0.044206, -0.02136, -0.043387, -0.028015, 0.0...",0.407789
8,My Python code works on the first try… said no...,"[0.038937, -0.032795, -0.005337, -0.001472, 0....",0.408450
9,The C# compiler walked into a bar. The bartend...,"[0.02272, -0.017781, -0.066756, -0.017234, 0.1...",0.411249


## Custom embedder

I want to use from nvidia, which exists in the documentation, but unfortunately it is not implemented. That's why I will implement my own custom embedder for nvidia nemotron

In [5]:
# OPTION B - our own embedding function, defined in custom_embeddings/nvidia_nemotron.py
# The import below does two things:
#   1. loads the class `NvidiaNemotronEmbedding`
#   2. its decorator @get_registry().register("nvidia-nemotron") adds it to the registry,
#      so get_registry().get("nvidia-nemotron").create() works afterwards.
# Needs NVIDIA_API in .env (read inside nvidia_nemotron.py).

from lancedb.pydantic import LanceModel, Vector
from custom_embeddings.nvidia_nemotron import NvidiaNemotronEmbedding
from lancedb.embeddings import get_registry 

embedding_model = get_registry().get("nvidia-nemotron").create()


# A NEW schema tied to the NVIDIA embedder. We cannot reuse JokeModel from option A: it is bound to the
# local sentence-transformers `model`, so a table built with it would never call NVIDIA.
class NvidiaJokeModel(LanceModel):
    jokes: str = embedding_model.SourceField()  # text goes in
    vector: Vector(embedding_model.ndims()) = embedding_model.VectorField()  # 4096 numbers come out


# mode="overwrite" replaces the table if it exists (re-run safe, but it deletes the old data).
table_jokes = db.create_table("jokes_nvidia", schema=NvidiaJokeModel, mode="overwrite")

# One NVIDIA API call per joke (see nvidia_nemotron.py), so this takes a little while.
table_jokes.add(df)


AddResult(version=2)

In [6]:
# Search the "jokes_nvidia" table: the query is embedded by NVIDIA (input_type="query"), then nearest jokes come back.
# "kemi" = chemistry in Danish/Swedish, again a multilingual test.
table_jokes.search("kemi").to_pandas()

,jokes,vector,_distance
0,Why did the chemist ground his kids? Because t...,"[0.000951, -0.075073, 0.017822, 0.00642, 0.010...",1.507765
1,I told a chemistry joke… there was no reaction.,"[0.005886, -0.082947, 0.019272, 0.000912, 0.02...",1.530094
2,Chemists have all the solutions… mostly in bea...,"[0.002228, -0.099121, 0.020493, 0.021805, 0.01...",1.582942
3,Never trust an atom—they make up everything.,"[0.010674, -0.074829, 0.017456, 0.009392, 0.01...",1.677488
4,Oxygen and Magnesium got together—OMg!,"[-0.011795, -0.058441, 0.016251, -0.004665, 0....",1.723530
5,"Gold walks into a bar. The bartender says, “Au...","[-0.004944, -0.056854, 0.010658, -0.011421, 0....",1.752608
6,My math teacher said I’m average… how mean!,"[0.010635, -0.061951, 0.010033, -0.00795, 0.02...",1.790349
7,"Dear algebra, please stop asking us to find yo...","[0.011032, -0.091125, 0.021072, 0.007797, 0.02...",1.790459
8,Why do data engineers hate nature? Too many un...,"[-0.006134, -0.076904, 0.011269, -0.00084, 0.0...",1.798832
9,Why was the equal sign so humble? It knew it w...,"[-0.008629, -0.060516, 0.003263, 0.012245, 0.0...",1.832485


In [7]:
# Same question as the e5-small search above, but on the NVIDIA table (table_jokes), so you can compare the two models.
table_jokes.search("why do mathematicians have problems?").to_pandas()

,jokes,vector,_distance
0,Chemists have all the solutions… mostly in bea...,"[0.002228, -0.099121, 0.020493, 0.021805, 0.01...",1.480776
1,"Dear algebra, please stop asking us to find yo...","[0.011032, -0.091125, 0.021072, 0.007797, 0.02...",1.620503
2,C# devs don’t argue; they just throw exceptions.,"[0.009048, -0.062347, 0.02359, 0.017288, 0.013...",1.655725
3,Why do data engineers hate nature? Too many un...,"[-0.006134, -0.076904, 0.011269, -0.00084, 0.0...",1.673184
4,My math teacher said I’m average… how mean!,"[0.010635, -0.061951, 0.010033, -0.00795, 0.02...",1.677725
5,Python devs don’t need therapy—they already im...,"[-0.016861, -0.0849, 0.016449, 0.013252, 0.007...",1.687831
6,Why did the chemist ground his kids? Because t...,"[0.000951, -0.075073, 0.017822, 0.00642, 0.010...",1.696849
7,Why did the Python programmer get bitten? Beca...,"[-0.013878, -0.042084, 0.011299, -0.019653, 0....",1.705387
8,Why did the C# developer go broke? He kept usi...,"[-0.010506, -0.058746, 0.014175, -0.005405, 0....",1.710946
9,Why’s 6 afraid of 7? Because 7 8 9.,"[-0.005344, -0.05307, 0.00255, -0.018265, 0.02...",1.722975
